In [1]:
import json
from pathlib import Path
from typing import List, Dict

In [2]:
def extract_annotations_from_inception(inception_json_path: Path) -> List[Dict]:
    """Extract PatristicReference annotations from INCEpTION JSON"""
    
    data = json.loads(inception_json_path.read_text(encoding="utf-8"))
    
    annotations = []
    
    # Find the sofa (document text)
    sofa = None
    for item in data.get("%FEATURE_STRUCTURES", []):
        if item.get("%TYPE") == "uima.cas.Sofa":
            sofa = item
            break
    
    if not sofa:
        return []
    
    sofa_id = sofa["%ID"]
    document_text = sofa.get("sofaString", "")
    
    # Extract PatristicReference annotations
    for item in data.get("%FEATURE_STRUCTURES", []):
        if item.get("%TYPE") == "webanno.custom.PatristicReference":
            
            # Extract text span
            begin = item["begin"]
            end = item["end"]
            letter_text = document_text[begin:end]
            
            annotations.append({
                "letter_chunk_id": item.get("letter_chunk_id"),
                "source_chunk_id": item.get("source_id"), 
                "letter_text": letter_text,
                "patristic_text": item.get("patristic_text", ""),
                "patristic_source": item.get("patristic_source", ""),
                "reference_type": item.get("reference_type", ""),
                "confidence": item.get("confidence", ""),
                "church_fathers": item.get("church_fathers", ""),
                "detection_source": item.get("detection_source", ""),
                "begin": begin,
                "end": end
            })
    
    return annotations


def process_all_inception_files(annotations_root: Path, output_dir: Path):
    """
    Process all INCEpTION annotation files and create validation dataset
    
    Expected structure:
    annotations_root/
      10294.txt/
        CURATION_USER2132954945288195931.json
      1888.txt/
        CURATION_USER2132954945288195931.json
      ...
    """
    
    all_annotations = []
    processed_letters = 0
    
    # Iterate through letter folders
    for letter_folder in sorted(annotations_root.iterdir()):
        if not letter_folder.is_dir():
            continue
        
        # Extract letter ID from folder name
        letter_id = letter_folder.name.replace(".txt", "")
        
        # Find the CURATION JSON file in this folder
        curation_files = list(letter_folder.glob("CURATION_USER*.json"))
        
        if not curation_files:
            print(f"No curation file found in {letter_folder.name}")
            continue
        
        json_file = curation_files[0]
        print(f"Processing {letter_id}...")
        
        annotations = extract_annotations_from_inception(json_file)
        
        # Add letter_id to each annotation
        for anno in annotations:
            anno["letter_id"] = letter_id
        
        all_annotations.extend(annotations)
        processed_letters += 1
    
    # Save validation dataset
    validation_data = {
        "metadata": {
            "total_annotations": len(all_annotations),
            "total_letters": processed_letters,
            "explicit_refs": sum(1 for a in all_annotations if a["reference_type"] == "explicit"),
            "implicit_refs": sum(1 for a in all_annotations if a["reference_type"] == "implicit"),
            "description": "Ground truth patristic references for validation"
        },
        "annotations": all_annotations
    }
    
    validation_path = output_dir / "validation_annotations.json"
    validation_path.write_text(
        json.dumps(validation_data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    
    print(f"Created validation dataset: {len(all_annotations)} annotations from {processed_letters} letters")
    print(f"  Saved to: {validation_path}")
    
    return all_annotations


def create_ir_ground_truth(validation_annotations: List[Dict], output_dir: Path):
    """
    Create ground truth in format:
    {
      "query_chunk_id": "10294_sent_13",
      "relevant_chunks": ["040_Augustinus_window_191"],
      "metadata": {...}
    }
    """
    
    ground_truth = []
    skipped = 0
    
    for anno in validation_annotations:
        if not anno.get("letter_chunk_id") or not anno.get("source_chunk_id"):
            print(f"Skipping annotation without chunk IDs in letter {anno.get('letter_id')}")
            skipped += 1
            continue
        
        ground_truth.append({
            "query_chunk_id": anno["letter_chunk_id"],
            "relevant_chunks": [anno["source_chunk_id"]],  # Single relevant chunk
            "letter_id": anno["letter_id"],
            "reference_type": anno["reference_type"],
            "confidence": anno["confidence"],
            "church_fathers": anno["church_fathers"]
        })
    
    output_data = {
        "metadata": {
            "description": "Ground truth for IR evaluation (chunk-level)",
            "total_queries": len(ground_truth),
            "skipped_annotations": skipped,
            "by_reference_type": {
                "explicit": sum(1 for gt in ground_truth if gt["reference_type"] == "explicit"),
                "implicit": sum(1 for gt in ground_truth if gt["reference_type"] == "implicit")
            }
        },
        "ground_truth": ground_truth
    }
    
    gt_path = output_dir / "ir_ground_truth.json"
    gt_path.write_text(
        json.dumps(output_data, ensure_ascii=False, indent=2),
        encoding="utf-8"
    )
    
    print(f"Created IR ground truth: {len(ground_truth)} query-document pairs")
    if skipped > 0:
        print(f"Skipped {skipped} annotations missing chunk IDs")
    print(f"Saved to: {gt_path}")


if __name__ == "__main__":
    
    # Paths
    annotations_root = Path("../annotations/annotations-json")
    output_dir = Path("../annotations/evaluation-dataset")

    
    # Step 1: Extract annotations from INCEpTION
    print("\n[1/2] Extracting annotations from INCEpTION exports...")
    annotations = process_all_inception_files(annotations_root, output_dir)
    
    # Step 2: Create IR ground truth
    print("\n[2/2] Creating IR ground truth...")
    create_ir_ground_truth(annotations, output_dir)
    

    print("✓ VALIDATION DATASET CREATED")

    print(f"\nSummary:")
    print(f"  Total annotations: {len(annotations)}")
    print(f"  Unique letters: {len(set(a['letter_id'] for a in annotations))}")
    print(f"  Explicit refs: {sum(1 for a in annotations if a['reference_type'] == 'explicit')}")
    print(f"  Implicit refs: {sum(1 for a in annotations if a['reference_type'] == 'implicit')}")


[1/2] Extracting annotations from INCEpTION exports...
Processing 10015...
Processing 10041...
Processing 10053...
Processing 10065...
Processing 10067...
Processing 10286...
Processing 10404...
Processing 10454...
Processing 10478...
Processing 10484...
Processing 10501...
Processing 10506...
Processing 10511...
Processing 10518...
Processing 10768...
Processing 10857...
Processing 10908...
Processing 10936...
Processing 11022...
Processing 11148...
Processing 11150...
Processing 11171...
Processing 11194...
Processing 11212...
Processing 11246...
Processing 11326...
Processing 11337...
Processing 11372...
Processing 11507...
Processing 11522...
Processing 11591...
Processing 11593...
Processing 11623...
Processing 11689...
Processing 11751...
Processing 11754...
Processing 11755...
Processing 11818...
Processing 11820...
Processing 11826...
Processing 11968...
Processing 11976...
Processing 11990...
Processing 12046...
Processing 12080...
Processing 12265...
Processing 12334...
Proc